# IT3051 - Fundamentals of Data Mining & Machine Learning
## Mini Project 2026: Used Car Price Prediction System
### Stage 6: Model Development & Stage 7: Optimization — Random Forest Regressor

**Assigned Model:** Random Forest Regressor (Ensemble Bagging)  
**Target Variable:** `log_price` (log-transformed vehicle selling price)  
**Input Features:** 13 engineered and scaled features from `Used_Car_Price_Prediction_System.ipynb`

In [ ]:
# 1. Import Core Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

pd.set_option('display.float_format', '{:,.4f}'.format)

---
## 2. Load Preprocessed Datasets

In [ ]:
# Load preprocessed datasets
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test  = pd.read_csv('y_test.csv').values.ravel()

print(f"X_train shape: {X_train.shape} (3,206 samples, 13 features)")
print(f"X_test  shape: {X_test.shape} (802 samples, 13 features)")

---
## 3. Baseline Random Forest & 5-Fold Cross-Validation
We train a baseline Random Forest Regressor (`n_estimators=100`, `random_state=42`) using **5-Fold Cross-Validation** on training data only.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
base_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

base_cv = cross_validate(
    base_rf,
    X_train,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

base_cv_r2   = base_cv['test_r2'].mean()
base_cv_r2_s = base_cv['test_r2'].std()
base_cv_mae  = -base_cv['test_mae'].mean()
base_cv_rmse = -base_cv['test_rmse'].mean()

# Fit baseline and evaluate on untouched test set
base_rf.fit(X_train, y_train)
y_pred_base = base_rf.predict(X_test)

base_test_r2   = r2_score(y_test, y_pred_base)
base_test_mae  = mean_absolute_error(y_test, y_pred_base)
base_test_rmse = root_mean_squared_error(y_test, y_pred_base)

y_test_dollar = np.expm1(y_test)
y_pred_base_dollar = np.expm1(y_pred_base)
base_test_mae_dollar  = mean_absolute_error(y_test_dollar, y_pred_base_dollar)
base_test_rmse_dollar = root_mean_squared_error(y_test_dollar, y_pred_base_dollar)

print("=== Baseline Random Forest Performance ===")
print(f"CV R²                 : {base_cv_r2:.4f} (+/- {base_cv_r2_s:.4f})")
print(f"CV MAE                : {base_cv_mae:.4f}")
print(f"CV RMSE               : {base_cv_rmse:.4f}")
print(f"Test R²               : {base_test_r2:.4f}")
print(f"Test MAE ($ scale)    : ${base_test_mae_dollar:,.2f}")
print(f"Test RMSE ($ scale)   : ${base_test_rmse_dollar:,.2f}")

---
## 4. Feature Importance Analysis
Random Forest Mean Decrease in Impurity (MDI) importances reflect each feature's contribution to variance reduction across the ensemble.
> **Viva Note:** Indicates predictive utility, **not causality**.

In [ ]:
# Feature importances extraction
importances = pd.Series(base_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

os.makedirs(os.path.join('results', 'feature_importance'), exist_ok=True)
rf_imp_df = pd.DataFrame({
    'Feature': importances.index,
    'Importance': importances.values,
    'Percentage': (importances.values * 100).round(2)
})
rf_imp_df.to_csv(os.path.join('results', 'feature_importance', 'random_forest_importance.csv'), index=False)

print("=== Random Forest Feature Importances ===")
print(rf_imp_df.to_string(index=False))

# Plot feature importance
plt.figure(figsize=(10, 5))
plt.barh(rf_imp_df['Feature'][::-1], rf_imp_df['Percentage'][::-1], color='steelblue', edgecolor='black')
plt.xlabel('Importance (%)')
plt.title('Random Forest Feature Importances (%)')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

---
## 5. Hyperparameter Tuning (Stage 7: Optimization)
We tune `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, and `max_features` using `RandomizedSearchCV` on the training set only.

In [ ]:
param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [None, 12, 18, 25],
    'min_samples_split': [2, 5, 8],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 0.8, 1.0]
}

search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=8,
    scoring='r2',
    cv=kf,
    random_state=42,
    n_jobs=-1
)
search.fit(X_train, y_train)

best_params = search.best_params_
print("=== Best Random Forest Hyperparameters ===")
print(f"Best Parameters: {best_params}")
print(f"Best Tuned CV R²: {search.best_score_:.4f}")

---
## 6. Tuned Random Forest Evaluation

In [ ]:
tuned_rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)

tuned_cv = cross_validate(
    tuned_rf,
    X_train,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

tuned_cv_r2   = tuned_cv['test_r2'].mean()
tuned_cv_r2_s = tuned_cv['test_r2'].std()
tuned_cv_mae  = -tuned_cv['test_mae'].mean()
tuned_cv_rmse = -tuned_cv['test_rmse'].mean()

tuned_rf.fit(X_train, y_train)
y_pred_tuned = tuned_rf.predict(X_test)

tuned_test_r2   = r2_score(y_test, y_pred_tuned)
tuned_test_mae  = mean_absolute_error(y_test, y_pred_tuned)
tuned_test_rmse = root_mean_squared_error(y_test, y_pred_tuned)

y_pred_tuned_dollar = np.expm1(y_pred_tuned)
tuned_test_mae_dollar  = mean_absolute_error(y_test_dollar, y_pred_tuned_dollar)
tuned_test_rmse_dollar = root_mean_squared_error(y_test_dollar, y_pred_tuned_dollar)

print("=== Tuned Random Forest Performance ===")
print(f"Tuned CV R²           : {tuned_cv_r2:.4f} (+/- {tuned_cv_r2_s:.4f})")
print(f"Tuned CV MAE          : {tuned_cv_mae:.4f}")
print(f"Tuned CV RMSE         : {tuned_cv_rmse:.4f}")
print(f"Tuned Test R²         : {tuned_test_r2:.4f}")
print(f"Tuned Test MAE ($)    : ${tuned_test_mae_dollar:,.2f}")
print(f"Tuned Test RMSE ($)   : ${tuned_test_rmse_dollar:,.2f}")

---
## 7. Feature Selection Experiment
We evaluate whether retaining only high-importance features (\(\ge 1.5\%\) importance) improves model generalization.

In [ ]:
threshold = 0.015
selected_feats = importances[importances >= threshold].index.tolist()
dropped_feats  = importances[importances < threshold].index.tolist()

print(f"Features retained ({len(selected_feats)}): {selected_feats}")
print(f"Features dropped  ({len(dropped_feats)}): {dropped_feats}")

X_train_sel = X_train[selected_feats]
X_test_sel  = X_test[selected_feats]

sel_rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
sel_cv = cross_validate(
    sel_rf,
    X_train_sel,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

sel_cv_r2   = sel_cv['test_r2'].mean()
sel_cv_r2_s = sel_cv['test_r2'].std()
sel_cv_mae  = -sel_cv['test_mae'].mean()
sel_cv_rmse = -sel_cv['test_rmse'].mean()

sel_rf.fit(X_train_sel, y_train)
y_pred_sel = sel_rf.predict(X_test_sel)

sel_test_r2   = r2_score(y_test, y_pred_sel)
sel_test_mae  = mean_absolute_error(y_test, y_pred_sel)
sel_test_rmse = root_mean_squared_error(y_test, y_pred_sel)

y_pred_sel_dollar = np.expm1(y_pred_sel)
sel_test_mae_dollar  = mean_absolute_error(y_test_dollar, y_pred_sel_dollar)
sel_test_rmse_dollar = root_mean_squared_error(y_test_dollar, y_pred_sel_dollar)

print("=== Feature Selection Experiment Results ===")
print(f"Selected CV R²        : {sel_cv_r2:.4f} (+/- {sel_cv_r2_s:.4f})")
print(f"Selected Test R²      : {sel_test_r2:.4f}")
print(f"Selected Test MAE ($) : ${sel_test_mae_dollar:,.2f}")
print(f"Selected Test RMSE ($): ${sel_test_rmse_dollar:,.2f}")
print("\nConclusion: Retaining all 13 features is superior; dropping secondary features lost predictive signal.")

---
## 8. Save Model Evaluation Records

In [ ]:
records = [
    {
        'Model': 'Random Forest',
        'Version': 'Baseline',
        'Parameters': 'n_estimators=100, max_depth=None',
        'CV_R2': base_cv_r2,
        'CV_R2_Std': base_cv_r2_s,
        'CV_MAE': base_cv_mae,
        'CV_RMSE': base_cv_rmse,
        'Test_R2': base_test_r2,
        'Test_MAE': base_test_mae,
        'Test_RMSE': base_test_rmse,
        'Test_MAE_Dollar': base_test_mae_dollar,
        'Test_RMSE_Dollar': base_test_rmse_dollar,
        'Observation': 'Solid bagging baseline; mitigates single-tree variance through ensemble averaging.'
    },
    {
        'Model': 'Random Forest',
        'Version': 'Tuned',
        'Parameters': str(best_params),
        'CV_R2': tuned_cv_r2,
        'CV_R2_Std': tuned_cv_r2_s,
        'CV_MAE': tuned_cv_mae,
        'CV_RMSE': tuned_cv_rmse,
        'Test_R2': tuned_test_r2,
        'Test_MAE': tuned_test_mae,
        'Test_RMSE': tuned_test_rmse,
        'Test_MAE_Dollar': tuned_test_mae_dollar,
        'Test_RMSE_Dollar': tuned_test_rmse_dollar,
        'Observation': 'Best Random Forest configuration. Constraining depth and features reduced variance.'
    },
    {
        'Model': 'Random Forest',
        'Version': 'Selected-Features (4 feats)',
        'Parameters': 'threshold>=0.015, ' + str(best_params),
        'CV_R2': sel_cv_r2,
        'CV_R2_Std': sel_cv_r2_s,
        'CV_MAE': sel_cv_mae,
        'CV_RMSE': sel_cv_rmse,
        'Test_R2': sel_test_r2,
        'Test_MAE': sel_test_mae,
        'Test_RMSE': sel_test_rmse,
        'Test_MAE_Dollar': sel_test_mae_dollar,
        'Test_RMSE_Dollar': sel_test_rmse_dollar,
        'Observation': 'Performance dropped. Dropping minor features lost collective predictive signal.'
    }
]

pd.DataFrame(records).to_csv(os.path.join('results', 'random_forest_results.csv'), index=False)
print("Random Forest evaluation saved to results/random_forest_results.csv")